# Aksara OCR — ablations on Kaggle

Runs the size and augmentation ablations, which the headline Colab session ran
out of time for. Kaggle suits them: they are independent, cheap per run, and —
unlike Colab-without-Drive — **resumable across sessions without any external
storage**.

### One-time setup (right sidebar → Settings)
1. **Accelerator → GPU P100.** Not "GPU T4 x2": the training loop is single-GPU,
   so the second T4 would sit idle. P100 also has tensor cores, so the AMP in
   these configs actually helps.
2. **Internet → On.** Required for the repo clone, pip, and the Mendeley fetch.

### How resume works (no Drive, no tokens)
Kaggle persists `/kaggle/working` when you **Save Version** (top right → *Save &
Run All*). To continue in a later session:
1. Save a version of this notebook.
2. In the next session, **+ Add Input → Your Work →** this notebook's output.
3. Run top to bottom. The restore cell copies the previous `results/` back in,
   and the runner skips every run that already finished.

Because runs are ordered seed-major and skipped when complete, each 12 h session
picks up exactly where the last stopped.

In [ ]:
import torch

assert torch.cuda.is_available(), \
    "No GPU. Settings (right sidebar) > Accelerator > GPU P100, then rerun."
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")
print(f"torch {torch.__version__}")

In [ ]:
# Internet must be ON (Settings > Internet) for these three lines.
import os
from pathlib import Path

REPO_URL = "https://github.com/phoenixfin/aksantara-ocr.git"
REPO = Path("/kaggle/working/aksantara-ocr")

if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}

os.chdir(REPO)
# torch/torchvision ship with Kaggle; installing the rest avoids a slow reinstall
# of torch against a possibly-mismatched CUDA build.
!pip install -q timm pyyaml scikit-image tabulate
print(f"ready: {Path.cwd()}")

In [ ]:
# Resume: if a previous version's output is attached as an input, copy its
# completed runs into the working artifacts dir so the runner skips them.
#
# Attach via: + Add Input (right sidebar) > Your Work > this notebook's output.
# Nothing to restore on the first session — this is a safe no-op then.
import shutil
from pathlib import Path

ARTIFACTS = Path("/kaggle/working/artifacts")
RESULTS = ARTIFACTS / "results" / "ablations"
RESULTS.mkdir(parents=True, exist_ok=True)

restored = 0
for candidate in Path("/kaggle/input").glob("*/artifacts/results/ablations"):
    for run_dir in candidate.iterdir():
        if not (run_dir / "result.json").exists():
            continue
        dest = RESULTS / run_dir.name
        if not dest.exists():
            shutil.copytree(run_dir, dest)
            restored += 1

done = len(list(RESULTS.glob("*/result.json")))
print(f"restored {restored} run(s) from a previous version; {done} complete in total")
if restored == 0:
    print("(first session, or no previous output attached — that is fine)")

In [ ]:
# Fetch the cleaned, published v3. ~808 MB; needs Internet ON.
DOI = "10.17632/vfj32bpjsf.3"
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --list-only
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --out /kaggle/working/raw

RAW_ROOT = "/kaggle/working/raw" 

In [ ]:
# Pre-resize once to 224px. Source images run up to 1500x1500; decoding one
# costs ~6.7 ms/core, so without this the dataloader, not the GPU, is the limit.
# A 224px cache drops that to ~1 ms/img.
!python scripts/00b_build_cache.py \
    --data-root "{RAW_ROOT}" --out /kaggle/working/data --size 224

DATA_ROOT = "/kaggle/working/data"
# Free the disk — the full-size tree is not needed again this session.
!rm -rf {RAW_ROOT}

In [ ]:
# stratified (no writer ids); --drop-duplicates removes images that became
# byte-identical after the 224px resize.
!python scripts/01_prepare_data.py \
    --data-root "{DATA_ROOT}" --out-dir "{ARTIFACTS}" \
    --split-strategy stratified --drop-duplicates

# Verify against published v3. Image/class/script counts come from manifest.csv,
# written before any filtering, so they are invariant.
import pandas as pd
manifest = pd.read_csv(f"{ARTIFACTS}/manifest.csv")
EXPECTED = {"images": 97383, "classes": 889, "scripts": 13}
actual = {"images": len(manifest), "classes": manifest["label"].nunique(),
          "scripts": manifest["script"].nunique()}
for k, want in EXPECTED.items():
    print(f"  {k:8} {actual[k]:6}  expected {want:6}  {'OK' if actual[k]==want else 'MISMATCH'}")
if actual != EXPECTED:
    raise SystemExit("Data does not match published v3 — re-run fetch and cache cells.")
print("Matches published v3.")

In [ ]:
# Both ablations write into one results dir so the report aggregates them, and
# so the shared center point (resnet18/unified/64px/medium/pretrained) — which
# is also the flat-895-class baseline to compare against the layered 98.5% —
# runs once.
#
# --num-workers 4: Kaggle gives 4 vCPUs (Colab gave 2).
# --time-budget 11: stop cleanly before the 12 h session limit; Save Version,
#   attach as input next time, and the rest resumes.
for cfg in ["configs/tier1_ablation_size.yaml", "configs/tier1_ablation_aug.yaml"]:
    print(f"\n{'='*70}\n{cfg}\n{'='*70}")
    !python scripts/02_run_matrix.py --config {cfg} \
        --artifacts "{ARTIFACTS}" --results "{RESULTS}" \
        --num-workers 4 --time-budget 11

In [ ]:
# CPU-only, no GPU time. Anchors the deep results against a non-deep baseline;
# if HOG+SVM lands close, that is itself a finding about dataset difficulty.
!python scripts/03_run_classical.py --artifacts "{ARTIFACTS}" 

In [ ]:
# Printed output is saved with the notebook version, so these tables survive
# even without downloading anything.
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)
report = RESULTS / "report"
for name in ["ablation_image_size", "ablation_augmentation", "ablation_pretrained",
             "main_benchmark"]:
    path = report / f"{name}.csv"
    if path.exists():
        print(f"\n=== {name} ===")
        print(pd.read_csv(path).to_string(index=False))

failures = RESULTS / "failures.jsonl"
if failures.exists():
    print(f"\nWARNING: some runs failed:\n{failures.read_text()[:2000]}")

## Finishing a session

Everything lives in `/kaggle/working/artifacts`. To keep it:

- **Save Version** (top right → *Save & Run All*, or *Quick Save* if the run is
  still going). That persists `/kaggle/working`.
- To download to your machine: **Output** tab → download `artifacts/`, or add
  the folder to your git checkout as `artifacts_kaggle/` and commit.

If the ablations did not all finish, start a new session, **+ Add Input → Your
Work →** this notebook's output, and run top to bottom — the restore cell brings
the finished runs back and the rest continues.